# Capstone Part 1: Pretrain a Language Model

In this notebook we build a ~10M parameter language model from scratch using modern architectural
components: **RoPE** (Rotary Position Embeddings), **RMSNorm**, **SwiGLU** activations, and
**Grouped-Query Attention (GQA)**. We then pretrain it on a subset of the TinyStories dataset.

By the end of this notebook you will have:
- A fully defined transformer architecture with state-of-the-art components
- A trained checkpoint saved to disk
- Training loss curves and sample generations

**Estimated runtime:** ~30-60 minutes on an 8GB GPU.

## 1. Environment Setup

We install and import all necessary libraries. The key dependencies are:
- `torch` for model definition and training
- `datasets` from Hugging Face for data loading
- `tiktoken` for a fast BPE tokenizer
- `matplotlib` for visualization

In [ ]:
!pip install -q torch datasets tiktoken matplotlib tqdm

In [ ]:
import math
import time
import json
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tiktoken
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Model Configuration

We define a configuration dataclass that controls every aspect of our model. The target is
~10M parameters, achieved with a relatively small hidden dimension and moderate depth.

Key design choices:
- **GQA** with fewer KV heads than query heads reduces memory during inference
- **SwiGLU** uses a gated linear unit with SiLU activation (Llama-style)
- **RMSNorm** is simpler and faster than LayerNorm
- **RoPE** provides relative position information without learned embeddings

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 50257       # GPT-2 tokenizer vocab size
    max_seq_len: int = 512
    d_model: int = 256            # Hidden dimension
    n_layers: int = 8             # Number of transformer blocks
    n_heads: int = 8              # Query heads
    n_kv_heads: int = 2           # Key/Value heads (GQA)
    d_ff: int = 688               # Feed-forward intermediate dim (SwiGLU: ~2.67 * d_model)
    dropout: float = 0.1
    rope_theta: float = 10000.0

config = ModelConfig()
print(f"Config: {config}")

## 3. RMSNorm

RMSNorm (Root Mean Square Layer Normalization) normalizes by the RMS of the activations
rather than computing full mean/variance like LayerNorm. This is both simpler and empirically
as effective. Used in Llama, Gemma, and most modern LLMs.

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot \gamma$$

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, dim)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float()).type_as(x)
        return output * self.weight


# Quick test
norm = RMSNorm(256)
test_input = torch.randn(2, 10, 256)
output = norm(test_input)
print(f"RMSNorm input shape: {test_input.shape}, output shape: {output.shape}")
print(f"Output RMS (should be ~1.0): {output.pow(2).mean(-1).sqrt().mean().item():.4f}")

## 4. Rotary Position Embeddings (RoPE)

RoPE encodes position by rotating query and key vectors in 2D subspaces. This yields:
- **Relative** position awareness (attention depends on distance, not absolute position)
- **Decaying** influence with distance (natural inductive bias)
- **Extrapolation** capability beyond training sequence length

For each pair of dimensions $(x_{2i}, x_{2i+1})$, we apply a rotation matrix with angle $m\theta_i$
where $m$ is the position and $\theta_i = \theta^{-2i/d}$.

In [ ]:
def precompute_rope_frequencies(dim: int, max_seq_len: int, theta: float = 10000.0):
    """Precompute the complex exponentials for RoPE."""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    positions = torch.arange(max_seq_len).float()
    # Outer product: (max_seq_len, dim//2)
    angles = torch.outer(positions, freqs)
    # Complex exponentials for rotation
    freqs_cis = torch.polar(torch.ones_like(angles), angles)  # e^(i*angle)
    return freqs_cis


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to input tensor.

    Args:
        x: (batch, seq_len, n_heads, head_dim)
        freqs_cis: (seq_len, head_dim//2)
    """
    # Reshape x to complex: treat pairs of dims as real/imag
    batch, seq_len, n_heads, head_dim = x.shape
    x_complex = torch.view_as_complex(x.float().reshape(batch, seq_len, n_heads, -1, 2))
    # Broadcast freqs_cis: (1, seq_len, 1, head_dim//2)
    freqs_cis = freqs_cis[:seq_len].unsqueeze(0).unsqueeze(2)
    # Apply rotation via complex multiplication
    x_rotated = torch.view_as_real(x_complex * freqs_cis).reshape(batch, seq_len, n_heads, head_dim)
    return x_rotated.type_as(x)


# Visualize the rotation frequencies
freqs_cis = precompute_rope_frequencies(config.d_model // config.n_heads, config.max_seq_len)
print(f"Precomputed RoPE frequencies shape: {freqs_cis.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
angles = freqs_cis.angle().numpy()
axes[0].imshow(angles[:64, :], aspect='auto', cmap='twilight')
axes[0].set_xlabel('Frequency index')
axes[0].set_ylabel('Position')
axes[0].set_title('RoPE Rotation Angles (first 64 positions)')
axes[0].colorbar = plt.colorbar(axes[0].images[0], ax=axes[0])

# Show how inner product decays with distance for different frequency bands
for dim_idx in [0, 4, 8, 12]:
    cos_vals = torch.cos(angles[:128, dim_idx] - angles[0, dim_idx])
    axes[1].plot(cos_vals, label=f'dim pair {dim_idx}')
axes[1].set_xlabel('Relative Position')
axes[1].set_ylabel('cos(relative angle)')
axes[1].set_title('Positional Decay by Frequency Band')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Grouped-Query Attention (GQA)

Standard multi-head attention uses separate K and V projections for each head, which is
expensive in memory (especially during inference with KV-cache). GQA groups multiple
query heads to share a single K/V head:

- **MHA** (Multi-Head Attention): `n_kv_heads == n_heads` (every Q head has its own KV)
- **MQA** (Multi-Query Attention): `n_kv_heads == 1` (all Q heads share one KV)
- **GQA**: `1 < n_kv_heads < n_heads` (groups of Q heads share KV)

We use `n_heads=8` query heads and `n_kv_heads=2` KV heads, so each KV head serves 4 query heads.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention with RoPE."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads  # How many Q heads per KV head

        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """Repeat KV heads to match the number of query heads.

        (batch, seq_len, n_kv_heads, head_dim) -> (batch, seq_len, n_heads, head_dim)
        """
        if self.n_rep == 1:
            return x
        batch, seq_len, n_kv_heads, head_dim = x.shape
        x = x[:, :, :, None, :].expand(batch, seq_len, n_kv_heads, self.n_rep, head_dim)
        return x.reshape(batch, seq_len, self.n_heads, head_dim)

    def forward(
        self, x: torch.Tensor, freqs_cis: torch.Tensor, mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        batch, seq_len, _ = x.shape

        # Project to Q, K, V
        q = self.wq(x).reshape(batch, seq_len, self.n_heads, self.head_dim)
        k = self.wk(x).reshape(batch, seq_len, self.n_kv_heads, self.head_dim)
        v = self.wv(x).reshape(batch, seq_len, self.n_kv_heads, self.head_dim)

        # Apply RoPE to Q and K
        q = apply_rope(q, freqs_cis)
        k = apply_rope(k, freqs_cis)

        # Repeat KV heads to match Q heads
        k = self._repeat_kv(k)
        v = self._repeat_kv(v)

        # Transpose for attention: (batch, n_heads, seq_len, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled dot-product attention
        scale = math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / scale

        if mask is not None:
            attn_weights = attn_weights.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        output = torch.matmul(attn_weights, v)  # (batch, n_heads, seq_len, head_dim)
        output = output.transpose(1, 2).reshape(batch, seq_len, -1)

        return self.resid_dropout(self.wo(output))


# Verify GQA parameter count
gqa = GroupedQueryAttention(config)
gqa_params = sum(p.numel() for p in gqa.parameters())
print(f"GQA parameters: {gqa_params:,}")
print(f"  Q projection: {config.d_model * config.n_heads * (config.d_model // config.n_heads):,}")
print(f"  K projection: {config.d_model * config.n_kv_heads * (config.d_model // config.n_heads):,}")
print(f"  V projection: {config.d_model * config.n_kv_heads * (config.d_model // config.n_heads):,}")
print(f"  Output projection: {config.n_heads * (config.d_model // config.n_heads) * config.d_model:,}")

## 6. SwiGLU Feed-Forward Network

SwiGLU is a gated feed-forward variant that outperforms standard ReLU FFNs:

$$\text{SwiGLU}(x) = (\text{SiLU}(xW_1) \odot xW_3) W_2$$

The gate ($W_3$) modulates the activated signal ($W_1$), providing finer-grained control.
Note that the intermediate dimension is typically $\frac{2}{3} \times 4d$ to keep the same
parameter count as a standard FFN with $4d$ intermediate size.

In [ ]:
class SwiGLU(nn.Module):
    """SwiGLU feed-forward network as used in Llama."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.w1 = nn.Linear(config.d_model, config.d_ff, bias=False)  # Gate projection
        self.w2 = nn.Linear(config.d_ff, config.d_model, bias=False)  # Down projection
        self.w3 = nn.Linear(config.d_model, config.d_ff, bias=False)  # Up projection
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SiLU(xW1) * xW3, then project down
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


ffn = SwiGLU(config)
ffn_params = sum(p.numel() for p in ffn.parameters())
print(f"SwiGLU parameters: {ffn_params:,}")
print(f"  3 projections of {config.d_model} x {config.d_ff}")

## 7. Transformer Block

Each block follows the **pre-norm** pattern used in modern LLMs:

```
x = x + Attention(RMSNorm(x))
x = x + FFN(RMSNorm(x))
```

Pre-norm (vs post-norm) improves training stability, especially at scale.

In [ ]:
class TransformerBlock(nn.Module):
    """Single transformer block with pre-norm, GQA, and SwiGLU."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model)
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config)

    def forward(
        self, x: torch.Tensor, freqs_cis: torch.Tensor, mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        # Pre-norm attention with residual
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        # Pre-norm FFN with residual
        x = x + self.ffn(self.ffn_norm(x))
        return x


block = TransformerBlock(config)
block_params = sum(p.numel() for p in block.parameters())
print(f"Transformer block parameters: {block_params:,}")
print(f"  Attention: {sum(p.numel() for p in block.attention.parameters()):,}")
print(f"  FFN: {sum(p.numel() for p in block.ffn.parameters()):,}")
print(f"  Norms: {sum(p.numel() for p in block.attention_norm.parameters()) + sum(p.numel() for p in block.ffn_norm.parameters()):,}")

## 8. Full Language Model

The complete model stacks all components:
1. Token embedding (no positional embedding -- RoPE handles positions)
2. N transformer blocks
3. Final RMSNorm
4. Linear head (weight-tied with embedding)

In [ ]:
class MiniLLM(nn.Module):
    """A ~10M parameter language model with modern architecture."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: share embedding and output weights
        self.lm_head.weight = self.tok_emb.weight

        # Precompute RoPE frequencies
        head_dim = config.d_model // config.n_heads
        self.register_buffer(
            "freqs_cis",
            precompute_rope_frequencies(head_dim, config.max_seq_len, config.rope_theta),
            persistent=False,
        )

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self, input_ids: torch.Tensor, targets: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch, seq_len = input_ids.shape

        # Create causal mask
        mask = torch.tril(torch.ones(seq_len, seq_len, device=input_ids.device))
        mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

        # Token embeddings (no positional embedding -- RoPE handles this)
        x = self.dropout(self.tok_emb(input_ids))

        # Transformer blocks
        for layer in self.layers:
            x = layer(x, self.freqs_cis, mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self, input_ids: torch.Tensor, max_new_tokens: int = 100,
        temperature: float = 0.8, top_k: int = 50
    ) -> torch.Tensor:
        """Autoregressive generation with top-k sampling."""
        for _ in range(max_new_tokens):
            # Crop to max_seq_len if needed
            idx_cond = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            # Top-k filtering
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids


model = MiniLLM(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (MB, fp32): {total_params * 4 / 1024 / 1024:.2f}")

## 9. Parameter Breakdown

Let's examine where the parameters live. In small models the embedding dominates;
in large models the transformer layers dominate.

In [ ]:
# Parameter breakdown by component
breakdown = {}
for name, param in model.named_parameters():
    component = name.split(".")[0]
    if component == "layers":
        # Further split by sub-component
        sub = name.split(".")[2]  # e.g., 'attention', 'ffn', 'attention_norm', 'ffn_norm'
        key = f"layers.{sub}"
    else:
        key = component
    breakdown[key] = breakdown.get(key, 0) + param.numel()

fig, ax = plt.subplots(figsize=(10, 6))
keys = list(breakdown.keys())
values = list(breakdown.values())
colors = plt.cm.Set3(range(len(keys)))
bars = ax.barh(keys, values, color=colors)
ax.set_xlabel("Number of Parameters")
ax.set_title("Parameter Distribution by Component")
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height() / 2,
            f"{val:,} ({val/total_params*100:.1f}%)", va='center')
plt.tight_layout()
plt.show()

## 10. Tokenizer Setup

We use `tiktoken` with the GPT-2 BPE tokenizer. This is fast and well-tested.
Our model's `vocab_size` of 50257 matches this tokenizer.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

# Quick demo
sample_text = "Once upon a time, a little cat went on an adventure."
tokens = tokenizer.encode(sample_text)
print(f"Text: {sample_text}")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"Decoded: {tokenizer.decode(tokens)}")
print(f"Token strings: {[tokenizer.decode([t]) for t in tokens]}")

## 11. Dataset: TinyStories

TinyStories is a dataset of short children's stories generated by GPT-3.5/4. It's ideal for
training small language models because:
- Stories use simple vocabulary and grammar
- They have clear narrative structure
- Small models can actually learn meaningful generation from this data

We load a subset and tokenize it into fixed-length chunks.

In [ ]:
# Load a subset of TinyStories
print("Loading TinyStories dataset...")
raw_dataset = load_dataset("roneneldan/TinyStories", split="train[:50000]")
print(f"Loaded {len(raw_dataset)} stories")
print(f"\nSample story:\n{raw_dataset[0]['text'][:500]}")

In [ ]:
class TinyStoriesDataset(Dataset):
    """Tokenized TinyStories dataset with fixed-length chunks."""

    def __init__(self, raw_dataset, tokenizer, seq_len: int = 512):
        self.seq_len = seq_len
        self.tokenizer = tokenizer

        # Tokenize all stories and concatenate
        print("Tokenizing dataset...")
        all_tokens = []
        for example in tqdm(raw_dataset, desc="Tokenizing"):
            tokens = tokenizer.encode(example["text"])
            all_tokens.extend(tokens)

        # Create fixed-length chunks
        self.tokens = torch.tensor(all_tokens, dtype=torch.long)
        self.n_chunks = len(self.tokens) // (seq_len + 1)  # +1 for target shift
        self.tokens = self.tokens[: self.n_chunks * (seq_len + 1)]
        self.tokens = self.tokens.reshape(self.n_chunks, seq_len + 1)
        print(f"Created {self.n_chunks:,} chunks of length {seq_len}")
        print(f"Total tokens: {len(all_tokens):,}")

    def __len__(self):
        return self.n_chunks

    def __getitem__(self, idx):
        chunk = self.tokens[idx]
        x = chunk[:-1]   # Input
        y = chunk[1:]    # Target (shifted by 1)
        return x, y


dataset = TinyStoriesDataset(raw_dataset, tokenizer, seq_len=config.max_seq_len)
print(f"\nDataset size: {len(dataset)} chunks")

In [ ]:
# Create train/val split
val_size = min(500, len(dataset) // 10)
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 12. Training Configuration

We use the standard recipe for small LLM pretraining:
- **AdamW** optimizer with weight decay (decoupled from learning rate)
- **Cosine learning rate schedule** with linear warmup
- **bf16 mixed precision** for faster training and lower memory
- **Gradient clipping** to prevent instability

In [ ]:
@dataclass
class TrainConfig:
    num_epochs: int = 3
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 100
    max_grad_norm: float = 1.0
    log_interval: int = 50
    eval_interval: int = 200
    save_dir: str = "checkpoints"

train_config = TrainConfig()

# Create save directory
save_dir = Path(train_config.save_dir)
save_dir.mkdir(exist_ok=True)

# Optimizer with weight decay only on 2D parameters (not biases, norms, embeddings)
decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
no_decay_params = [p for n, p in model.named_parameters() if p.dim() < 2 and p.requires_grad]

optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": train_config.weight_decay},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=train_config.learning_rate, betas=(0.9, 0.95))

print(f"Optimizer groups:")
print(f"  Decay params: {sum(p.numel() for p in decay_params):,}")
print(f"  No-decay params: {sum(p.numel() for p in no_decay_params):,}")

In [ ]:
def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps):
    """Cosine learning rate schedule with linear warmup."""
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))  # Min LR = 10% of peak

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


total_steps = len(train_loader) * train_config.num_epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, train_config.warmup_steps, total_steps)
print(f"Total training steps: {total_steps}")

# Visualize LR schedule
lrs = []
temp_scheduler = get_cosine_schedule_with_warmup(
    torch.optim.AdamW(model.parameters(), lr=train_config.learning_rate),
    train_config.warmup_steps, total_steps
)
for step in range(total_steps):
    lrs.append(temp_scheduler.get_last_lr()[0])
    temp_scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("Cosine Schedule with Warmup")
plt.grid(True, alpha=0.3)
plt.show()

## 13. Evaluation Function

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader, device):
    """Compute validation loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_batches = 0

    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            _, loss = model(x, y)
        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / total_batches
    perplexity = math.exp(avg_loss)
    model.train()
    return avg_loss, perplexity

## 14. Training Loop

The main training loop with:
- Mixed precision (bf16) for speed
- Gradient accumulation support
- Periodic evaluation and logging
- Checkpoint saving

In [ ]:
# Training state tracking
train_losses = []
val_losses = []
val_perplexities = []
learning_rates = []
step_times = []
global_step = 0
best_val_loss = float("inf")

# Mixed precision scaler
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler(enabled=use_amp)

print(f"Starting training for {train_config.num_epochs} epochs ({total_steps} steps)")
print(f"Mixed precision: {use_amp}")
print("=" * 70)

model.train()
for epoch in range(train_config.num_epochs):
    epoch_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        step_start = time.time()
        x, y = x.to(device), y.to(device)

        # Forward pass with mixed precision
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            logits, loss = model(x, y)

        # Backward pass
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.max_grad_norm)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        # Track metrics
        train_losses.append(loss.item())
        learning_rates.append(scheduler.get_last_lr()[0])
        step_times.append(time.time() - step_start)
        epoch_loss += loss.item()
        global_step += 1

        # Log
        if global_step % train_config.log_interval == 0:
            avg_step_time = sum(step_times[-train_config.log_interval:]) / train_config.log_interval
            tokens_per_sec = (32 * config.max_seq_len) / avg_step_time
            print(
                f"  Step {global_step}/{total_steps} | "
                f"Loss: {loss.item():.4f} | "
                f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                f"Tok/s: {tokens_per_sec:.0f}"
            )

        # Evaluate
        if global_step % train_config.eval_interval == 0:
            val_loss, val_ppl = evaluate(model, val_loader, device)
            val_losses.append((global_step, val_loss))
            val_perplexities.append((global_step, val_ppl))
            print(f"  >> Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "step": global_step,
                    "val_loss": val_loss,
                }, save_dir / "best_pretrained.pt")
                print(f"  >> Saved best checkpoint (val_loss={val_loss:.4f})")

    epoch_time = time.time() - epoch_start
    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete | Avg Loss: {avg_epoch_loss:.4f} | Time: {epoch_time:.1f}s")
    print("=" * 70)

# Save final checkpoint
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config,
    "step": global_step,
    "train_losses": train_losses,
    "val_losses": val_losses,
}, save_dir / "final_pretrained.pt")
print("\nTraining complete! Final checkpoint saved.")

## 15. Training Curves

Visualize how loss and perplexity evolved during training.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training loss (smoothed)
ax = axes[0, 0]
ax.plot(train_losses, alpha=0.2, color='blue', label='Raw')
# Exponential moving average
ema_losses = []
ema = train_losses[0]
for loss in train_losses:
    ema = 0.99 * ema + 0.01 * loss
    ema_losses.append(ema)
ax.plot(ema_losses, color='blue', label='EMA (0.99)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss
ax = axes[0, 1]
if val_losses:
    val_steps, val_loss_vals = zip(*val_losses)
    ax.plot(val_steps, val_loss_vals, 'ro-', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.grid(True, alpha=0.3)

# Perplexity
ax = axes[1, 0]
if val_perplexities:
    val_steps, val_ppl_vals = zip(*val_perplexities)
    ax.plot(val_steps, val_ppl_vals, 'go-', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Perplexity')
ax.set_title('Validation Perplexity')
ax.grid(True, alpha=0.3)

# Learning rate
ax = axes[1, 1]
ax.plot(learning_rates, color='orange')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)

plt.suptitle('Training Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'pretrain_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to checkpoints/pretrain_curves.png")

## 16. Sample Generations from the Pretrained Model

Let's see what our freshly pretrained model can generate. At this stage the model
should produce coherent-ish text that resembles simple stories, but it has no
instruction-following ability yet.

In [ ]:
model.eval()

prompts = [
    "Once upon a time",
    "The little dog",
    "One sunny morning",
    "There was a big",
]

print("=" * 70)
print("SAMPLE GENERATIONS (Pretrained Model)")
print("=" * 70)

for prompt in prompts:
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    output_ids = model.generate(input_ids, max_new_tokens=80, temperature=0.8, top_k=40)
    generated_text = tokenizer.decode(output_ids[0].tolist())
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 70)

## 17. Token Distribution Analysis

Let's look at the model's output distribution to verify it has learned meaningful patterns.

In [ ]:
# Analyze top predicted tokens for a given context
test_prompt = "The cat sat on the"
input_ids = torch.tensor([tokenizer.encode(test_prompt)], device=device)

with torch.no_grad():
    logits, _ = model(input_ids)
    probs = F.softmax(logits[0, -1, :], dim=-1)

top_k = 20
top_probs, top_indices = torch.topk(probs, top_k)

print(f"Context: '{test_prompt}'")
print(f"\nTop {top_k} predicted next tokens:")
tokens_labels = []
probs_values = []
for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
    token_str = tokenizer.decode([idx.item()])
    tokens_labels.append(repr(token_str))
    probs_values.append(prob.item())
    print(f"  {i+1:2d}. {repr(token_str):15s} prob={prob.item():.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(range(top_k), probs_values[::-1])
ax.set_yticks(range(top_k))
ax.set_yticklabels(tokens_labels[::-1])
ax.set_xlabel('Probability')
ax.set_title(f'Next Token Predictions for: "{test_prompt}"')
plt.tight_layout()
plt.show()

## 18. Checkpoint Summary

Let's verify our saved checkpoint and summarize the pretraining results.

In [ ]:
# Verify checkpoint can be loaded
checkpoint = torch.load(save_dir / "best_pretrained.pt", map_location=device, weights_only=False)
print("Checkpoint contents:")
print(f"  Config: {checkpoint['config']}")
print(f"  Step: {checkpoint['step']}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")
print(f"  Val Perplexity: {math.exp(checkpoint['val_loss']):.2f}")
print(f"  State dict keys: {len(checkpoint['model_state_dict'])}")

# File sizes
for f in save_dir.glob("*.pt"):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

## Summary

In this notebook we:

1. **Defined** a ~10M parameter LLM with modern components (RoPE, RMSNorm, SwiGLU, GQA)
2. **Loaded** and tokenized the TinyStories dataset
3. **Trained** with AdamW, cosine schedule, and bf16 mixed precision
4. **Saved** checkpoints for the best and final models
5. **Visualized** training curves and sample generations

The pretrained model can generate coherent story-like text but has no instruction-following
ability. In the next notebook (02-sft.ipynb) we will fine-tune it on instruction data.

**Checkpoint location:** `checkpoints/best_pretrained.pt`